# Full-cohort calibration comparison

Companion to the submitted spectral stress-testing study. Select a GPU runtime before inference. Run the cells in order. Published records remain inside the uploaded package; new outputs go to a separate Drive project.

## Install analysis/inference dependencies

Keep the CUDA-compatible torch and torchvision supplied by the runtime. See the README for the recorded software environment.

In [ ]:
import sys, subprocess

def run_live(command):
    print("Running:", " ".join(map(str, command)), flush=True)
    with subprocess.Popen(command, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)
    print("Completed successfully.", flush=True)

run_live([sys.executable, "-m", "pip", "install", "-q",
          "timm==1.0.28", "numpy", "pandas", "scipy",
          "Pillow", "matplotlib", "threadpoolctl", "tqdm"])


## Upload the supplementary archive and run CPU checks

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, sys, subprocess
uploaded = files.upload()
archives = [Path(name) for name in uploaded if name.endswith(".zip")]
if len(archives) != 1:
    raise ValueError("Upload only the complete reproducibility ZIP or Supplementary_Code_and_Results.zip.")
destination = Path("/content/cviu_repro")
destination.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as archive:
    for item in archive.infolist():
        resolved = (destination / item.filename).resolve()
        if not resolved.is_relative_to(destination.resolve()):
            raise ValueError("Unsafe archive member.")
    archive.extractall(destination)
candidates = [p for p in destination.iterdir() if p.is_dir() and (p / "prepare_replay_project.py").is_file() and (p / "strengthening_experiments").is_dir()]
if len(candidates) != 1:
    raise ValueError("Extract one complete reproducibility archive into an empty destination.")
PACKAGE = candidates[0]
run_live([sys.executable, str(PACKAGE / "strengthening_experiments/cpu_checks.py")])


## Select a writable replay project

The default folder is separate from the original experiment folders. You may change PROJECT before continuing.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/CVIU_Spectral_Replay")
print("New/replay project:", PROJECT, flush=True)
print("Package:", PACKAGE, flush=True)


## Stage frozen inputs

This step regenerates the nine fitting-view logit caches on GPU; it does not refit the saved calibrators.

In [ ]:
command = [sys.executable, str(PACKAGE / "prepare_replay_project.py"),
           "--project", str(PROJECT)]
command.append("--regenerate-fit-logits")
run_live(command)


## Benchmark

This is a timing check, not a paper result. Read the console output; fitting, scoring and bootstrap take additional time.

In [ ]:
run_live([sys.executable, str(PACKAGE / "strengthening_experiments/run_strengthening.py"),
                "--project", str(PROJECT), "--task", "calibration", "--mode", "benchmark",
                "--batch-size", "24", "--workers", "2"])


## Full run

Run after the benchmark succeeds. If disconnected, rerun this cell with the same project and settings to resume verified shards. Do not compare timing from cached full runs with a fresh benchmark.

In [ ]:
run_live([sys.executable, str(PACKAGE / "strengthening_experiments/run_strengthening.py"),
                "--project", str(PROJECT), "--task", "calibration", "--mode", "full",
                "--batch-size", "24", "--workers", "2"])


## Outputs

The runner prints the run folder and a DONE line after each job. Summaries, compact predictions, logits, timing, and analysis outputs are under PROJECT/runs/. A completed full run has RUN_COMPLETE.json and paired-analysis files.